a motivating question for why your are building the dataset you are building

Do colleges with more students, housing, and closer to cities cost more to attend? I chose to look specifically at rhode island
colleges.code and description of how you built and prepared your dataset. For each step, describe what you’re about to do, the code with output, interpretation that leads into the next step. exploratory data analysis that shows 


## Web Scraping

In [4]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

In [5]:
#import url, use beautiful soup to read the html from the file I provided and store it a beautifulSoup variable
college_url = 'https://nces.ed.gov/collegenavigator/?s=RI&l=93'
college_html = requests.get(college_url)
college_data = BeautifulSoup(college_html.content, 'html.parser')
#I looked on the websites inspect feature to see that some general information about each college in the table is
#stored in a class called 'iBack'
colleges = college_data.find_all('div','iBack')
#created empty lists to the data i will scrape about each website
names=[]
population=[]
housing=[]
setting = []
#looped through each college in the table and saw that the name of each one is stored in 'h2'. 
for college in colleges:
    names.append(college.find('h2').text.strip())
    #saw on the websites inspect that each row of the table is stored in 'tr'
    rows = college.find_all('tr')
    for row in rows:
        #stored the info into a variable to make sure it is not empty. again I found 'td' and 'srb' by using inspect
        info = row.find('td', 'srb')
        if info:
            #because a lot of information was stored under the same section ('td','sra'), i had to go through each row and compare
            #it to a string to see which row has what information. If found then It gets appended to the corresponding list
            if "population" in info.text.lower():
                population.append(row.find('td','sra').text.strip())
            elif "housing" in info.text.lower():
                housing.append(row.find('td', 'sra').text.strip())
            elif "setting" in info.text.lower():
                setting.append(row.find('td', 'sra').text.strip()) 
#created dataframe and gave each list a column
colleges_df = pd.DataFrame({'name':names,'population':population,
              'setting':setting, 'housing':housing})

In [6]:
#output: show the dataframe I created. Becuase there are not many colleges in RI i printed the whole df rather than just the head
colleges_df

,name,population,setting,housing
0,Brown University,"11,189 (7,639 undergraduate)",City: Midsize,Yes
1,Bryant University,"3,536 (3,204 undergraduate)",Suburb: Large,Yes
2,College Unbound,285 (all undergraduate),City: Midsize,No
3,Johnson & Wales University-Online,"2,505 (1,843 undergraduate)",City: Midsize,No
4,Johnson & Wales University-Providence,"4,566 (4,158 undergraduate)",City: Midsize,Yes
5,New England Institute of Technology,"1,895 (1,720 undergraduate)",Suburb: Large,Yes
6,Providence College,"4,473 (3,996 undergraduate)",City: Midsize,Yes
7,Rhode Island College,"5,787 (4,719 undergraduate)",Suburb: Large,Yes
8,Rhode Island School of Design,"2,620 (2,108 undergraduate)",City: Midsize,Yes
9,Roger Williams University,"4,397 (4,103 undergraduate)",Suburb: Large,Yes


## Crawling

In [53]:
#i created a new variable looking for 'tr' with both classes 'resultW' and 'resultY' since each result on the website
#alternated between these two classes (for some reason that I dont know)
college_items2 = college_data.find_all('tr',['resultsW','resultsY']) 
#empty list for tuitions of each college
tuitions = []
#loop through each college
for current_college in college_items2:
    #Programatically find the url that leads to more information about the current college we are looking at.
    #I got 'a' and 'href' by inspecting the website
    path = current_college.find('a', href=True).attrs
    path = path['href']
    #Made sure the path is not empty
    if path:
        #had to add the first part of the url (not programatically) since the website only has the path for each college
        #rather than the entire link
        tuition_url = 'https://nces.ed.gov/collegenavigator/'+ path
        tuition_html = requests.get(tuition_url).content
        tuition_info = BeautifulSoup(tuition_html, 'html.parser')
        try: 
            #similar to above, the tuition did not have a unique class, it was lsited with a lot of other information 
            #so I had to loop through all the rows in the table and find which ones contained the header 'tuition and fees'
            all_td = tuition_info.find_all('td')
            for current_td in all_td:
                if 'Tuition and fees' in current_td:
                    #using the __dict__ finction, i saw that once we found the 'tuition and fees' header, the actual tuition
                    #cost was listed under 'next_sibling'
                    tuition = (current_td.next_sibling).contents
                    #appended tuition[0] since tuition got stored as a list with one element, rather than a string
                    tuitions.append(tuition[0])
        except:
            tuitions.append(pd.NA)
colleges_df['tuition ($)'] = tuitions

['$60,696', '$46,863', '$10,112', '$13,365', '$34,736', '$31,545', '$54,388', ' ', '$54,890', '$38,274', '$42,920', ' ']


In [54]:
colleges_df

,name,population,setting,housing,tuition ($)
0,Brown University,"11,189 (7,639 undergraduate)",City: Midsize,Yes,"$60,696"
1,Bryant University,"3,536 (3,204 undergraduate)",Suburb: Large,Yes,"$46,863"
2,College Unbound,285 (all undergraduate),City: Midsize,No,"$10,112"
3,Johnson & Wales University-Online,"2,505 (1,843 undergraduate)",City: Midsize,No,"$13,365"
4,Johnson & Wales University-Providence,"4,566 (4,158 undergraduate)",City: Midsize,Yes,"$34,736"
5,New England Institute of Technology,"1,895 (1,720 undergraduate)",Suburb: Large,Yes,"$31,545"
6,Providence College,"4,473 (3,996 undergraduate)",City: Midsize,Yes,"$54,388"
7,Rhode Island College,"5,787 (4,719 undergraduate)",Suburb: Large,Yes,
8,Rhode Island School of Design,"2,620 (2,108 undergraduate)",City: Midsize,Yes,"$54,890"
9,Roger Williams University,"4,397 (4,103 undergraduate)",Suburb: Large,Yes,"$38,274"


In [57]:
#In order to better analyze the data, I want to change tution to an integer 
#so I can further explore it to answer my motivating question

#function to convert all instances of tuition to integers by dropping the the $ and commas
def convert_tuition(s):
    if pd.notna(s) and s != ' ':
        s = str(s)
        s = s.replace('$', '').replace(',', '')
        return int(s)
    else:
        return pd.NA

#apply the conversion function to the 'tuition' column
#replace the old tuition comma with strings to the new one with integers
colleges_df['tuition ($)'] = colleges_df['tuition ($)'].apply(convert_tuition)

#displaying updated dataframe with the replaced tuition column
colleges_df

,name,population,setting,housing,tuition ($)
0,Brown University,"11,189 (7,639 undergraduate)",City: Midsize,Yes,60696
1,Bryant University,"3,536 (3,204 undergraduate)",Suburb: Large,Yes,46863
2,College Unbound,285 (all undergraduate),City: Midsize,No,10112
3,Johnson & Wales University-Online,"2,505 (1,843 undergraduate)",City: Midsize,No,13365
4,Johnson & Wales University-Providence,"4,566 (4,158 undergraduate)",City: Midsize,Yes,34736
5,New England Institute of Technology,"1,895 (1,720 undergraduate)",Suburb: Large,Yes,31545
6,Providence College,"4,473 (3,996 undergraduate)",City: Midsize,Yes,54388
7,Rhode Island College,"5,787 (4,719 undergraduate)",Suburb: Large,Yes,<NA>
8,Rhode Island School of Design,"2,620 (2,108 undergraduate)",City: Midsize,Yes,54890
9,Roger Williams University,"4,397 (4,103 undergraduate)",Suburb: Large,Yes,38274


In [59]:
#output: updated df with a new column called tuition that holds the cost of tuition for each college. where
#tuition was found programatically
colleges_df

,name,population,setting,housing,tuition ($)
0,Brown University,"11,189 (7,639 undergraduate)",City: Midsize,Yes,60696
1,Bryant University,"3,536 (3,204 undergraduate)",Suburb: Large,Yes,46863
2,College Unbound,285 (all undergraduate),City: Midsize,No,10112
3,Johnson & Wales University-Online,"2,505 (1,843 undergraduate)",City: Midsize,No,13365
4,Johnson & Wales University-Providence,"4,566 (4,158 undergraduate)",City: Midsize,Yes,34736
5,New England Institute of Technology,"1,895 (1,720 undergraduate)",Suburb: Large,Yes,31545
6,Providence College,"4,473 (3,996 undergraduate)",City: Midsize,Yes,54388
7,Rhode Island College,"5,787 (4,719 undergraduate)",Suburb: Large,Yes,<NA>
8,Rhode Island School of Design,"2,620 (2,108 undergraduate)",City: Midsize,Yes,54890
9,Roger Williams University,"4,397 (4,103 undergraduate)",Suburb: Large,Yes,38274


## Saving to CSV

In [12]:
colleges_df.to_csv('college_information.csv')

## Exploritory data analysis

why you blt the data and confirms that is prepared enough to analyze.

In [15]:
colleges_df.describe()

,name,population,setting,housing,tuition ($)
count,12,12,12,12,10
unique,12,12,2,2,10
top,Brown University,"11,189 (7,639 undergraduate)",City: Midsize,Yes,60696
freq,1,1,6,10,1


There are 12 total unique colleges in Rhode Island. What is the most common setting for RI colleges?

In [17]:
colleges_df['setting'].mode()

0    City: Midsize
1    Suburb: Large
Name: setting, dtype: object

We can see that the most frequent setting is in a city with a midsize campus, followed by suburbs with a large campus. 
This information would be helpful for oncoming college students who are still not sure which school to go to.
They can use their personal preferences along with the table I created to try and decide.
This ties back to my original motivating question, which is trying to find the relationship between cost and population, setting, and housing. By comparing the settings and tuition, I want to find out whether colleges that are more suburban or city have a higher or lower tuition.

In [19]:
#I cleaned my dataset by converting tuition to an integer, which now lets me use the mean function to compare it to
#other categories. This is one example of my data being cleaned enough to be able to analyze it.
colleges_df.groupby('setting')['tuition ($)'].mean()

setting
City: Midsize    38031.166667
Suburb: Large         39900.5
Name: tuition ($), dtype: object

we can see that 'city:midsize' colleges have a somewhat lower tuition. This helps answer my motivating question since someone can use this comparison to understand what setting of college they would want to go to depending on their budget for tuition.

In [21]:
colleges_df.groupby('housing')['tuition ($)'].mean()

housing
No     11738.5
Yes    45539.0
Name: tuition ($), dtype: object

Here, we see that colleges that do not offer housing have a significantlly lower tuition. This may help someone decide whether they should commute to college if they are trying to save money on tuition, which does help answer my motivating question.

In [23]:
population_df = colleges_df.groupby('population')['tuition ($)'].mean()
population_df

population
1,895 (1,720 undergraduate)      31545.0
11,189 (7,639 undergraduate)     60696.0
17,473 (14,768 undergraduate)        NaN
2,505 (1,843 undergraduate)      13365.0
2,620 (2,108 undergraduate)      54890.0
2,872 (2,190 undergraduate)      42920.0
285 (all undergraduate)          10112.0
3,536 (3,204 undergraduate)      46863.0
4,397 (4,103 undergraduate)      38274.0
4,473 (3,996 undergraduate)      54388.0
4,566 (4,158 undergraduate)      34736.0
5,787 (4,719 undergraduate)          NaN
Name: tuition ($), dtype: object

This doesnt tell us much since it is hard to see which populations have a higher/lower tuition.
I want to sort by tuition in this case.

In [25]:
population_df.sort_values()

population
285 (all undergraduate)          10112.0
2,505 (1,843 undergraduate)      13365.0
1,895 (1,720 undergraduate)      31545.0
4,566 (4,158 undergraduate)      34736.0
4,397 (4,103 undergraduate)      38274.0
2,872 (2,190 undergraduate)      42920.0
3,536 (3,204 undergraduate)      46863.0
4,473 (3,996 undergraduate)      54388.0
2,620 (2,108 undergraduate)      54890.0
11,189 (7,639 undergraduate)     60696.0
17,473 (14,768 undergraduate)        NaN
5,787 (4,719 undergraduate)          NaN
Name: tuition ($), dtype: object

As we can see, the colleges with the lowest tuitions have a significantly cheaper tuition. In fact, the smallest population also corresponds with the lowert tuition. This can help someone decide on a college with a bigger or smaller population depending on their budget for tuition.